# 잔차 패턴 분류범주 실험 — 고정 5-Fold / A LightGBM · B CatBoost
**실험 코드만 작성한 버전입니다. 실제 모델 학습·임계값 탐색은 실행하지 않았습니다.**

두 PART 모두 기존 Train 616,293행, Validation 133,316행, `event_id → fold_num`, `row_id`를 유지합니다.
각 예측 Fold마다 나머지 4개 Fold에서 회귀 4회 → 분류 1회로 예측범주를 만듭니다.
총 25회 + 최종 실험 회귀 1회가 핵심 26회입니다. Validation 분류용 회귀 5회·분류 1회,
기준 회귀 1회를 별도로 더하면 단일 임계값 전체는 **33회**입니다.

**이전 공통 1~6을 이미 실행했다면 아래 `공통 4 — 재실행 시작`부터 실행하세요.**
공통 1~3의 변수·입력·고정 Fold 검사는 유지했고 기존 함수 정의 셀은 제거했습니다.
PART A와 B 중 필요한 PART만 실행할 수 있습니다. 두 PART 사이에는 잔차·모델 공유가 없습니다.

A에서 Validation 점수를 보고 임계값을 선택하므로 이 Validation은 이번에는 **모델 선택에도 사용되는 평가셋**입니다.
B의 Validation 점수는 새로운 미사용 Test 성능으로 해석하지 않습니다. 단, 어떤 fit/조기 종료에도 Validation 정답은 넣지 않습니다.


In [ ]:
# 공통 1. 라이브러리

from pathlib import Path
from datetime import datetime           # 실험 실행 시각 기록
from uuid import uuid4                  # 실험별 고유 ID 생성
import hashlib                          # 데이터 및 Fold 정합성 검증용 해시
import json                             # 실험 설정값 JSON 파일 저장
import warnings                         # 경고 메시지 출력 및 관리
import platform                         # Python 실행 환경 정보 확인
import importlib.metadata               # 설치된 패키지 버전 확인

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from catboost import CatBoostRegressor, CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import mean_squared_error, f1_score, confusion_matrix


In [ ]:
# 공통 2. 실제 기존 경로를 기본값으로 사용: 업로드한 (1) 파일은 명시된 별칭만 허용
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'parqueted_data'
CONFIG_DIR = PROJECT_ROOT / 'config'
OUTPUT_ROOT = Path.cwd() / 'residual_class_experiments'

# 원래 프로젝트에서 실행하면 위 경로 사용. 첨부 파일과 함께 실행할 때 아래 후보 사용.
# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet')

train_kf_oof = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_kf_oof_train2_index_fixed.parquet')

kf_fold_map = pd.read_csv(PROJECT_ROOT / 'config' / '0_kf_event_fold_match.csv')


In [ ]:
RANDOM_STATE = 42
REG_ITERATIONS = 3000
EARLY_STOPPING_ROUNDS = 100
THREAD_COUNT = -1
CLASSES = ['under', 'normal', 'over']
NEW_FEATURE = 'pred_residual_class'  # 이번 실험에서 새로 만드는 예측범주 열
# 분류기 설정은 기존 회귀 하이퍼파라미터가 아닌 신규 실험 설정입니다.
LGB_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=31,
                  random_state=42, n_jobs=THREAD_COUNT, verbosity=-1,
                  deterministic=True, force_col_wise=True)
CAT_CLS_PARAMS = dict(iterations=1000, random_state=42, loss_function='MultiClass',
                      thread_count=THREAD_COUNT, allow_writing_files=False, verbose=False)


In [ ]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")


In [ ]:
# 공통 3. 원본은 읽기만 수행; 행 순서를 바꾸는 merge/reset_index/sort 금지
for name, df in [('train', train), ('val', val), ('기존 OOF', train_kf_oof)]:
    assert df.index.name == 'row_id', f'{name}: row_id 인덱스 필요'
    assert df.index.is_unique and not df.index.hasnans, f'{name}: 중복/결측 row_id'
    assert 'row_id' not in df.columns
assert train.index.equals(train_kf_oof.index), 'Train/기존 OOF 행 순서 불일치: 자동 정렬하지 않음'
assert train.index.intersection(val.index).empty
assert set(train['event_id']).isdisjoint(set(val['event_id']))
assert kf_fold_map.columns.tolist() == ['event_id', 'fold_num']
assert not kf_fold_map.isna().any().any() and kf_fold_map['event_id'].is_unique
assert set(kf_fold_map['fold_num']) == {1, 2, 3, 4, 5}
assert set(train['event_id']) == set(kf_fold_map['event_id'])
assert not train['event_id'].isna().any() and not val['event_id'].isna().any()

kf_groups = train['event_id']
kf_fold_num = kf_groups.map(kf_fold_map.set_index('event_id')['fold_num']).astype(int)
assert kf_fold_num.notna().all()
assert np.array_equal(kf_fold_num.to_numpy(), train_kf_oof['fold_num'].to_numpy())
assert (kf_fold_num.groupby(kf_groups).nunique() == 1).all()
assert np.array_equal(train[target_col].to_numpy(), train_kf_oof['y_kf_val'].to_numpy())
# 기존 OOF의 원래 입력 열도 현재 Train과 완전히 같은지 확인
pd.testing.assert_frame_equal(train, train_kf_oof[train.columns])
assert np.isfinite(train_kf_oof[['y_kf_pred', 'resid', 'resid_sq']].to_numpy()).all()
assert np.allclose(train_kf_oof['resid'], train_kf_oof['y_kf_val'] - train_kf_oof['y_kf_pred'])
assert np.allclose(train_kf_oof['resid_sq'], train_kf_oof['resid'] ** 2)

X_train_with_station = train[num_cols_fe + ['station_id']].copy()
X_val_with_station = val[num_cols_fe + ['station_id']].copy()
for X in [X_train_with_station, X_val_with_station]:
    assert X['station_id'].notna().all()
    X['station_id'] = X['station_id'].astype(str)
    assert not np.isinf(X[num_cols_fe].to_numpy(dtype=float)).any()
    assert all(pd.api.types.is_numeric_dtype(X[c]) for c in num_cols_fe)
y_train = train[target_col]
assert np.isfinite(y_train.to_numpy()).all()
assert len(num_cols_fe) == 31 and X_train_with_station.shape[1] == 32

# 원본 확인용 스냅샷: 실험 종료 때 다시 비교
original_row_index = train.index.copy()
original_fold_num = kf_fold_num.copy()
old_oof_rmse = float(np.sqrt(mean_squared_error(y_train, train_kf_oof['y_kf_pred'])))
print(f'Train={train.shape}, Validation={val.shape}, 고정 Fold=5')
print(f'저장된 기존 OOF RMSE={old_oof_rmse:.10f} (새 평가의 기준값으로 대입하지 않음)')
display(pd.crosstab(kf_fold_num, columns='행 수'))


## 공통 4 — 재실행 시작
기존 커널에서는 여기부터 실행합니다. 이미 불러온 데이터와 X/y/Fold를 재사용하며 파일을 다시 읽지 않습니다.
이 셀은 LightGBM 회귀 import, A의 회귀 설정, 데이터 서명을 준비합니다. **def/클래스 정의는 없습니다.**
데이터·Fold·독립변수를 변경했다면 공통 3까지 다시 준비한 후 이 셀과 해당 PART의 준비 셀을 다시 실행하세요.


In [ ]:
# 공통 4 — 재실행 시작: 기존 공통 1~6 실행 상태와 새 커널 모두 지원
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

# A 신규 LightGBM 회귀 설정: 내부 잔차 회귀와 기준/실험 회귀에 같은 설정 적용
A_REG_PARAMS = dict(n_estimators=3000, learning_rate=0.05, num_leaves=31,
                    random_state=RANDOM_STATE, n_jobs=THREAD_COUNT,
                    verbosity=-1, deterministic=True, force_col_wise=True)
# B 기존 CatBoost 회귀 하이퍼파라미터 유지
B_REG_PARAMS = dict(iterations=REG_ITERATIONS, random_state=RANDOM_STATE,
                    thread_count=THREAD_COUNT, allow_writing_files=False)
assert REG_ITERATIONS == 3000 and EARLY_STOPPING_ROUNDS == 100
assert len(train) == 616293 and len(val) == 133316
assert train.index.equals(X_train_with_station.index) and train.index.equals(y_train.index)
assert val.index.equals(X_val_with_station.index)
assert kf_fold_num.index.equals(train.index)
assert set(kf_fold_num.unique()) == {1, 2, 3, 4, 5}
assert np.array_equal(kf_fold_num.to_numpy(),
    train['event_id'].map(kf_fold_map.set_index('event_id')['fold_num']).to_numpy())
assert (kf_fold_num.groupby(train['event_id']).nunique() == 1).all()
assert train.index.intersection(val.index).empty
assert set(train['event_id']).isdisjoint(set(val['event_id']))
assert X_train_with_station.columns.tolist() == num_cols_fe + ['station_id']
assert X_val_with_station.columns.equals(X_train_with_station.columns)
assert NEW_FEATURE == 'pred_residual_class' and NEW_FEATURE not in X_train_with_station

# 데이터/전처리 변경 시 잔차 재생성 필요. Validation 정답은 서명에도 사용하지 않음.
data_signature = hashlib.sha256(
    pd.util.hash_pandas_object(X_train_with_station, index=True).to_numpy().tobytes()
    + pd.util.hash_pandas_object(y_train, index=True).to_numpy().tobytes()
    + pd.util.hash_pandas_object(kf_fold_num, index=True).to_numpy().tobytes()
    + pd.util.hash_pandas_object(X_val_with_station, index=True).to_numpy().tobytes()
).hexdigest()
class_dtype = pd.CategoricalDtype(categories=CLASSES, ordered=False)
print('기존 고정 5-Fold / row_id / 32개 입력 확인 완료. 모델 학습은 각 PART 셀에서 수행됩니다.')


## PART A — 전부 LightGBM
### A1. Train 분류 OOF용 회귀 잔차 준비 · 20회
예측 Fold A를 제외하면 B·C·D·E 각각을 내부 검증으로 한 번씩 사용합니다.
회귀의 학습 Fold는 내부 검증까지 제외한 3개입니다. A는 fit/eval_set 양쪽에서 제외됩니다.
이 셀은 Fold별 잔차를 메모리의 `a_nested_residuals`에 보관합니다.
논리적으로 각 Fold의 회귀 4회 뒤에 분류 1회가 이어지지만, 임계값 변경 시 회귀를 재학습하지 않도록
분류는 A5 셀에 분리했습니다. **임계값만 바꿀 때 이 셀은 다시 실행하지 않습니다.**


In [ ]:
# A1. 기존 고정 Fold를 그대로 순회: 5개 예측 Fold × 내부 회귀 4회
a_nested_residuals = {}
a_reg_log = []
a_cache_signature = data_signature
a_reg_snapshot = json.dumps(A_REG_PARAMS, sort_keys=True)
a_early_snapshot = EARLY_STOPPING_ROUNDS
for outer_fold in [1, 2, 3, 4, 5]:
    outer_ids = train.index[kf_fold_num.to_numpy() == outer_fold]
    remaining_ids = train.index[kf_fold_num.to_numpy() != outer_fold]
    inner_pred = pd.Series(np.nan, index=remaining_ids)
    inner_seen = pd.Series(0, index=remaining_ids)
    for inner_fold in [f for f in [1, 2, 3, 4, 5] if f != outer_fold]:
        # outer_fold는 예측 대상, inner_fold는 잔차 생성 대상, 나머지 3개만 학습
        fit_ids = train.index[(kf_fold_num.to_numpy() != outer_fold) &
                              (kf_fold_num.to_numpy() != inner_fold)]
        inner_ids = train.index[kf_fold_num.to_numpy() == inner_fold]
        assert fit_ids.intersection(inner_ids).empty
        assert fit_ids.intersection(outer_ids).empty and inner_ids.intersection(outer_ids).empty
        assert fit_ids.intersection(val.index).empty and inner_ids.intersection(val.index).empty
        assert set(kf_groups.loc[fit_ids]).isdisjoint(set(kf_groups.loc[inner_ids]))
        assert set(kf_groups.loc[remaining_ids]).isdisjoint(set(kf_groups.loc[outer_ids]))
        print(f'A 제외 Fold={outer_fold} / 내부 검증 Fold={inner_fold} / 나머지 3개 학습', flush=True)
        # LightGBM 범주 사전은 해당 회귀 학습 행에서만 생성
        X_fit = X_train_with_station.loc[fit_ids].copy()
        X_inner = X_train_with_station.loc[inner_ids].copy()
        station_dtype = pd.CategoricalDtype(pd.unique(X_fit['station_id']), ordered=False)
        X_fit['station_id'] = X_fit['station_id'].astype(station_dtype)
        X_inner['station_id'] = X_inner['station_id'].astype(station_dtype)
        model_reg = LGBMRegressor(**A_REG_PARAMS)
        model_reg.fit(X_fit, y_train.loc[fit_ids], eval_set=[(X_inner, y_train.loc[inner_ids])],
                      eval_metric='rmse', categorical_feature=['station_id'],
                      callbacks=[early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(0)])
        inner_pred.loc[inner_ids] = model_reg.predict(X_inner)
        tree_count = int(model_reg.best_iteration_)
        inner_seen.loc[inner_ids] += 1
        a_reg_log.append(dict(stage='nested_residual', outer_fold=outer_fold,
                                  inner_fold=inner_fold, fit_rows=len(fit_ids), trees=tree_count))
    assert (inner_seen == 1).all() and np.isfinite(inner_pred.to_numpy()).all()
    assert inner_pred.index.equals(remaining_ids)
    a_nested_residuals[outer_fold] = (y_train.loc[remaining_ids] - inner_pred).rename('residual')
assert len(a_nested_residuals) == 5 and len(a_reg_log) == 20
print('A1 완료: 회귀 20회. 임계값만 변경할 때는 재실행하지 않습니다.')


### A2. Validation 분류기 정답용 회귀 OOF · 추가 5회
Train 전체의 고정 5-Fold 회귀 OOF 잔차를 별도로 준비합니다.
Validation 타깃은 사용하지 않으며, 이 잔차도 임계값 변경 시 재사용합니다.
기존 `train_kf_oof` 잔차는 새 LightGBM 분류기 정답으로 대입하지 않습니다.


In [ ]:
# A2. 독립 Validation 분류기 학습용 정답 잔차 생성; 회귀 5회 추가
assert a_cache_signature == data_signature
assert a_reg_snapshot == json.dumps(A_REG_PARAMS, sort_keys=True)
assert a_early_snapshot == EARLY_STOPPING_ROUNDS
a_full_oof = pd.Series(np.nan, index=train.index)
full_seen = pd.Series(0, index=train.index)
a_full_reg_log = []
for inner_fold in [1, 2, 3, 4, 5]:
    fit_ids = train.index[kf_fold_num.to_numpy() != inner_fold]
    inner_ids = train.index[kf_fold_num.to_numpy() == inner_fold]
    assert fit_ids.intersection(inner_ids).empty
    assert fit_ids.intersection(val.index).empty and inner_ids.intersection(val.index).empty
    assert set(kf_groups.loc[fit_ids]).isdisjoint(set(kf_groups.loc[inner_ids]))
    inner_pred = pd.Series(np.nan, index=inner_ids)
    # LightGBM 범주 사전은 해당 회귀 학습 행에서만 생성
    X_fit = X_train_with_station.loc[fit_ids].copy()
    X_inner = X_train_with_station.loc[inner_ids].copy()
    station_dtype = pd.CategoricalDtype(pd.unique(X_fit['station_id']), ordered=False)
    X_fit['station_id'] = X_fit['station_id'].astype(station_dtype)
    X_inner['station_id'] = X_inner['station_id'].astype(station_dtype)
    model_reg = LGBMRegressor(**A_REG_PARAMS)
    model_reg.fit(X_fit, y_train.loc[fit_ids], eval_set=[(X_inner, y_train.loc[inner_ids])],
                  eval_metric='rmse', categorical_feature=['station_id'],
                  callbacks=[early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(0)])
    inner_pred.loc[inner_ids] = model_reg.predict(X_inner)
    tree_count = int(model_reg.best_iteration_)
    a_full_oof.loc[inner_ids] = inner_pred
    full_seen.loc[inner_ids] += 1
    a_full_reg_log.append(dict(stage='validation_label_residual', inner_fold=inner_fold,
                                   fit_rows=len(fit_ids), trees=tree_count))
assert (full_seen == 1).all() and np.isfinite(a_full_oof.to_numpy()).all()
a_full_residual = (y_train - a_full_oof).rename('residual')
a_full_signature = (data_signature, json.dumps(A_REG_PARAMS, sort_keys=True), EARLY_STOPPING_ROUNDS)
print('A2 완료: 추가 회귀 5회. 회귀 성능 진단이 아닌 분류 정답 생성용입니다.')


### A3. 신규 변수 없는 기준 회귀 · 추가 1회
전체 Train으로 학습하고 Validation **예측값만** 준비합니다. eval_set/조기 종료는 없습니다.
동일 모델 설정으로 임계값별 실험 회귀와 비교합니다. 임계값이 바뀌어도 기준 예측은 재사용합니다.


In [ ]:
# A3. 기준 LightGBM 회귀 1회: Validation 정답 미사용
assert a_full_signature == (data_signature, json.dumps(A_REG_PARAMS, sort_keys=True), EARLY_STOPPING_ROUNDS)
a_X_train = X_train_with_station.copy()
a_X_val = X_val_with_station.copy()
station_dtype = pd.CategoricalDtype(pd.unique(a_X_train['station_id']), ordered=False)
a_X_train['station_id'] = a_X_train['station_id'].astype(station_dtype)
a_X_val['station_id'] = a_X_val['station_id'].astype(station_dtype)
a_baseline_model = LGBMRegressor(**A_REG_PARAMS)
a_baseline_model.fit(a_X_train, y_train, categorical_feature=['station_id'])
a_baseline_pred = pd.Series(a_baseline_model.predict(a_X_val), index=val.index)
assert a_baseline_pred.index.equals(val.index) and np.isfinite(a_baseline_pred.to_numpy()).all()
a_baseline_signature = (data_signature, json.dumps(A_REG_PARAMS, sort_keys=True), EARLY_STOPPING_ROUNDS)
print('A3 기준 예측 준비 완료. 여기까지 공통 회귀 26회입니다.')


### A4. 임계값 후보 직접 입력
임계값만 변경할 때는 이 셀부터 다시 실행하세요. A1~A3 잔차·기준 예측을 재사용합니다.


In [ ]:
# A4. 사용자 지정 후보: 확정값이 아니며 자동 선택하지 않음
thresholds = [0.5, 1.0]
assert len(thresholds) > 0 and len(set(thresholds)) == len(thresholds)
assert all(np.isfinite(t) and t > 0 for t in thresholds)


### A5. Fold별 분류 → 예측범주 → Validation 비교
각 Fold의 분류기는 A1의 **해당 Fold를 제외한 회귀 4회 결과**로 학습합니다.
5개 Fold의 예측범주는 `row_id` 순서로 합칩니다. 이후 A2의 전체 회귀 OOF 잔차로
별도 분류기를 학습하여 Validation 범주를 만듭니다.

표의 기본 범주 비율은 A2의 Train 전체 회귀 OOF 정답 분포이며, 외부 Fold별 학습 정답 분포와
Train/Validation 예측범주 분포도 각각 저장합니다. Macro F1·혼동행렬의 정답은
**Validation 관측값 − 해당 PART 기준 회귀 예측값**을 라벨링한 평가용 값입니다.
이 정답은 모델 입력으로 사용하지 않습니다. 없는 범주도 포함한 고정 3범주 Macro F1입니다.


In [ ]:
# A5. 회귀 잔차는 재사용; 분류/실험 회귀만 임계값마다 학습
# 데이터 또는 설정을 바꿨으면 공통 4와 A1~A3을 먼저 다시 실행
current_signature = hashlib.sha256(
    pd.util.hash_pandas_object(X_train_with_station, index=True).to_numpy().tobytes()
    + pd.util.hash_pandas_object(y_train, index=True).to_numpy().tobytes()
    + pd.util.hash_pandas_object(kf_fold_num, index=True).to_numpy().tobytes()
    + pd.util.hash_pandas_object(X_val_with_station, index=True).to_numpy().tobytes()
).hexdigest()
assert current_signature == data_signature == a_cache_signature, '데이터 변경: 준비 셀 재실행 필요'
assert a_reg_snapshot == json.dumps(A_REG_PARAMS, sort_keys=True)
assert a_early_snapshot == EARLY_STOPPING_ROUNDS
assert a_full_signature == a_baseline_signature == (data_signature, json.dumps(A_REG_PARAMS, sort_keys=True), EARLY_STOPPING_ROUNDS)
assert train.index.equals(original_row_index)
assert np.array_equal(kf_fold_num.to_numpy(), original_fold_num.to_numpy())
assert np.array_equal(kf_fold_num.to_numpy(), train['event_id'].map(kf_fold_map.set_index('event_id')['fold_num']))

# 매 실행마다 새 폴더: 임계값을 바꿔도 과거 결과를 덮어쓰지 않음
a_dir = OUTPUT_ROOT / ('PART_A_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8])
a_dir.mkdir(parents=True, exist_ok=False)
(a_dir / 'settings.json').write_text(json.dumps(dict(
    thresholds=thresholds, regression_params=A_REG_PARAMS, classifier_params=LGB_PARAMS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS, data_signature=data_signature,
    versions={p: importlib.metadata.version(p) for p in ['pandas','scikit-learn','lightgbm','catboost']}),
    ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame(a_reg_log + a_full_reg_log).to_csv(a_dir / 'regression_training_log.csv', index=False)
a_results = []
a_classification_log = []
for threshold in thresholds:
    # B는 단일 확정값 하나만 수행; A만 후보를 반복 비교
    print(f'PART A: threshold={threshold}', flush=True)
    try:
        threshold_dir = a_dir / f'threshold_{threshold}'
        threshold_dir.mkdir(exist_ok=False)
        train_category = pd.Series(index=train.index, dtype=object, name=NEW_FEATURE)
        category_seen = pd.Series(0, index=train.index)
        fold_distributions = []
        for outer_fold in [1, 2, 3, 4, 5]:
            # 이 Fold의 분류 정답은 A1에서 이 Fold를 제외한 내부 회귀 4회로 생성됨
            predict_ids = train.index[kf_fold_num.to_numpy() == outer_fold]
            fit_ids = train.index[kf_fold_num.to_numpy() != outer_fold]
            residual = a_nested_residuals[outer_fold]
            assert residual.index.equals(fit_ids) and np.isfinite(residual.to_numpy()).all()
            assert fit_ids.intersection(predict_ids).empty and fit_ids.intersection(val.index).empty
            assert set(kf_groups.loc[fit_ids]).isdisjoint(set(kf_groups.loc[predict_ids]))
            residual_label = pd.Series(np.where(residual > threshold, 'under',
                np.where(residual < -threshold, 'over', 'normal')), index=fit_ids, name='residual_label')
            counts = residual_label.value_counts().reindex(CLASSES, fill_value=0)
            for label in CLASSES:
                fold_distributions.append(dict(fold_num=outer_fold, label=label,
                    count=int(counts[label]), ratio=float(counts[label] / len(fit_ids))))
            # 학습 정답이 한 범주라면 분류 fit을 건너뛰고 상수 예측, 실제 횟수 기록
            class_count = residual_label.nunique()
            if class_count == 1:
                warnings.warn(f'PART A Fold {outer_fold}: 한 범주만 존재, 상수 예측')
                predicted = np.repeat(residual_label.iloc[0], len(predict_ids))
            else:
                if class_count < 3:
                    warnings.warn(f'PART A Fold {outer_fold}: 학습 정답에 일부 범주가 없습니다.')
                X_fit = X_train_with_station.loc[fit_ids].copy()
                X_predict = X_train_with_station.loc[predict_ids].copy()
                station_dtype = pd.CategoricalDtype(pd.unique(X_fit['station_id']), ordered=False)
                X_fit['station_id'] = X_fit['station_id'].astype(station_dtype)
                X_predict['station_id'] = X_predict['station_id'].astype(station_dtype)
                model_cls = LGBMClassifier(**LGB_PARAMS)
                model_cls.fit(X_fit, residual_label, categorical_feature=['station_id'])
                predicted = np.asarray(model_cls.predict(X_predict)).reshape(-1).astype(str)
            predicted_category = pd.Series(predicted, index=predict_ids, name=NEW_FEATURE)
            assert set(predicted_category.unique()).issubset(CLASSES)
            train_category.loc[predict_ids] = predicted_category
            category_seen.loc[predict_ids] += 1
            a_classification_log.append(dict(threshold=float(threshold), fold=outer_fold,
                                                   fit_performed=class_count > 1))
        assert (category_seen == 1).all() and train_category.notna().all()
        assert train_category.index.equals(train.index) and train_category.name == NEW_FEATURE

        # Validation용 별도 분류기: A2의 Train 전체 회귀 OOF 잔차를 정답으로 사용
        residual = a_full_residual
        assert residual.index.equals(train.index) and np.isfinite(residual.to_numpy()).all()
        residual_label = pd.Series(np.where(residual > threshold, 'under',
            np.where(residual < -threshold, 'over', 'normal')), index=train.index, name='residual_label')
        counts = residual_label.value_counts().reindex(CLASSES, fill_value=0)
        distribution = pd.DataFrame({'count': counts, 'ratio': counts / len(train)})
        display(distribution.rename(index={'under':'과소예측','normal':'일반오차','over':'과대예측'}))
        class_count = residual_label.nunique()
        if class_count == 1:
            warnings.warn('Validation용 분류기 정답이 한 범주뿐입니다. 상수 예측 사용.')
            val_predicted = np.repeat(residual_label.iloc[0], len(val))
        else:
            if class_count < 3:
                warnings.warn('Validation용 분류기 학습 정답에 일부 범주가 없습니다.')
            model_val_cls = LGBMClassifier(**LGB_PARAMS)
            model_val_cls.fit(a_X_train, residual_label, categorical_feature=['station_id'])
            val_predicted = np.asarray(model_val_cls.predict(a_X_val)).reshape(-1).astype(str)
        a_classification_log.append(dict(threshold=float(threshold), fold='Validation',
                                              fit_performed=class_count > 1))
        val_category = pd.Series(val_predicted, index=val.index, name=NEW_FEATURE)
        assert val_category.notna().all() and set(val_category.unique()).issubset(CLASSES)
        assert val_category.index.equals(val.index) and val_category.name == NEW_FEATURE

        # 기존 32개 변수 + 분류 predict 결과 1개만 사용. residual_label은 입력하지 않음
        assert train_category.name == val_category.name == NEW_FEATURE
        X_exp_train = a_X_train.copy()
        X_exp_val = a_X_val.copy()
        X_exp_train[NEW_FEATURE] = train_category.astype(class_dtype)
        X_exp_val[NEW_FEATURE] = val_category.astype(class_dtype)
        assert X_exp_train[NEW_FEATURE].notna().all() and X_exp_val[NEW_FEATURE].notna().all()
        model_exp = LGBMRegressor(**A_REG_PARAMS)
        model_exp.fit(X_exp_train, y_train, categorical_feature=['station_id', NEW_FEATURE])
        experiment_pred = pd.Series(model_exp.predict(X_exp_val), index=val.index)
        assert X_exp_train.columns.tolist() == num_cols_fe + ['station_id', NEW_FEATURE]
        assert X_exp_val.columns.equals(X_exp_train.columns)
        assert X_exp_train.index.equals(y_train.index)
        assert experiment_pred.index.equals(val.index) and np.isfinite(experiment_pred.to_numpy()).all()

        # 모든 fit/predict 후에만 Validation 정답을 평가에 사용
        y_val = val[target_col]
        assert np.isfinite(y_val.to_numpy()).all()
        baseline_rmse = float(np.sqrt(mean_squared_error(y_val, a_baseline_pred)))
        experiment_rmse = float(np.sqrt(mean_squared_error(y_val, experiment_pred)))
        eval_residual = y_val - a_baseline_pred
        eval_label = pd.Series(np.where(eval_residual > threshold, 'under',
            np.where(eval_residual < -threshold, 'over', 'normal')), index=val.index)
        score_f1 = float(f1_score(eval_label, val_category, labels=CLASSES, average='macro', zero_division=0))
        matrix = pd.DataFrame(confusion_matrix(eval_label, val_category, labels=CLASSES), index=CLASSES, columns=CLASSES)
        print('Validation 혼동행렬: 행=정답, 열=예측')
        display(matrix)
        a_results.append(dict(threshold=float(threshold), under_count=int(counts['under']),
            normal_count=int(counts['normal']), over_count=int(counts['over']),
            under_ratio=float(distribution.loc['under','ratio']), normal_ratio=float(distribution.loc['normal','ratio']),
            over_ratio=float(distribution.loc['over','ratio']), macro_f1=score_f1,
            baseline_rmse=baseline_rmse, experiment_rmse=experiment_rmse,
            delta_rmse=experiment_rmse-baseline_rmse, improvement_rmse=baseline_rmse-experiment_rmse,
            improvement_pct=100*(baseline_rmse-experiment_rmse)/baseline_rmse if baseline_rmse else np.nan))
        # row_id 유지; 정답 분포와 예측 분포를 구분해 저장
        distribution.to_csv(threshold_dir / 'train_full_oof_label_distribution.csv')
        pd.DataFrame(fold_distributions).to_csv(threshold_dir / 'fold_training_label_distribution.csv', index=False)
        pd.DataFrame({'fold_num':kf_fold_num, NEW_FEATURE:train_category}).to_parquet(threshold_dir / 'train_classification_oof.parquet')
        pd.DataFrame({'y_true':y_val, 'baseline_pred':a_baseline_pred, 'experiment_pred':experiment_pred,
                       NEW_FEATURE:val_category, 'evaluation_label':eval_label}).to_parquet(threshold_dir / 'validation_predictions.parquet')
        # 범주별 데이터 수/비율: Train 예측, Validation 예측, Validation 평가 정답 각각 기록
        for name, labels in [('train_predicted',train_category), ('validation_predicted',val_category), ('validation_truth',eval_label)]:
            class_counts = labels.value_counts().reindex(CLASSES, fill_value=0)
            pd.DataFrame({'count':class_counts, 'ratio':class_counts/len(labels)}).to_csv(threshold_dir / f'{name}_distribution.csv')
        matrix.to_csv(threshold_dir / 'validation_confusion_matrix.csv')
        pd.DataFrame(a_results).to_csv(a_dir / 'validation_results.csv', index=False)
        pd.DataFrame(a_classification_log).to_csv(a_dir / 'classification_training_log.csv', index=False)
    except Exception as exc:
        (a_dir / 'error.txt').write_text(f'threshold={threshold}\n{type(exc).__name__}: {exc}', encoding='utf-8')
        raise
a_results_df = pd.DataFrame(a_results)
display(a_results_df)
assert train.index.equals(original_row_index)
assert np.array_equal(kf_fold_num.to_numpy(), original_fold_num.to_numpy())
print('ΔRMSE=실험−기준: 음수이면 개선. improvement_rmse=기준−실험: 양수이면 개선.')
print('결과 폴더:', a_dir)


In [ ]:
# A6. 후보 입력 순서대로 그래프 표시; 최적 임계값 자동 선택 없음
fig, ax = plt.subplots(figsize=(8, 4))
positions = np.arange(len(a_results_df))
ax.plot(positions, a_results_df['experiment_rmse'], marker='o', label='With predicted class')
ax.plot(positions, a_results_df['baseline_rmse'], linestyle='--', label='LightGBM baseline')
ax.set_xticks(positions, [str(t) for t in a_results_df['threshold']])
ax.set(xlabel='User-supplied residual threshold', ylabel='Validation RMSE')
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(a_dir / 'threshold_rmse.png', dpi=160)
plt.show()


## PART B — 전부 CatBoost
### B0. 확정 임계값 직접 입력
A에서 선택한 값은 사용자가 직접 입력합니다. A의 LightGBM 잔차·범주를 읽지 않습니다.


In [ ]:
# B0. 사용자가 직접 확정한 단일 임계값 입력
final_threshold = None
if final_threshold is None:
    raise ValueError('final_threshold에 직접 확정한 양의 임계값을 입력하세요.')
assert np.isfinite(final_threshold) and final_threshold > 0


### B1. Train 분류 OOF용 회귀 잔차 준비 · 20회
예측 Fold A를 제외하면 B·C·D·E 각각을 내부 검증으로 한 번씩 사용합니다.
회귀의 학습 Fold는 내부 검증까지 제외한 3개입니다. A는 fit/eval_set 양쪽에서 제외됩니다.
이 셀은 Fold별 잔차를 메모리의 `b_nested_residuals`에 보관합니다.
논리적으로 각 Fold의 회귀 4회 뒤에 분류 1회가 이어지지만, 임계값 변경 시 회귀를 재학습하지 않도록
분류는 B5 셀에 분리했습니다. **임계값만 바꿀 때 이 셀은 다시 실행하지 않습니다.**


In [ ]:
# B1. 기존 고정 Fold를 그대로 순회: 5개 예측 Fold × 내부 회귀 4회
b_nested_residuals = {}
b_reg_log = []
b_cache_signature = data_signature
b_reg_snapshot = json.dumps(B_REG_PARAMS, sort_keys=True)
b_early_snapshot = EARLY_STOPPING_ROUNDS
for outer_fold in [1, 2, 3, 4, 5]:
    outer_ids = train.index[kf_fold_num.to_numpy() == outer_fold]
    remaining_ids = train.index[kf_fold_num.to_numpy() != outer_fold]
    inner_pred = pd.Series(np.nan, index=remaining_ids)
    inner_seen = pd.Series(0, index=remaining_ids)
    for inner_fold in [f for f in [1, 2, 3, 4, 5] if f != outer_fold]:
        # outer_fold는 예측 대상, inner_fold는 잔차 생성 대상, 나머지 3개만 학습
        fit_ids = train.index[(kf_fold_num.to_numpy() != outer_fold) &
                              (kf_fold_num.to_numpy() != inner_fold)]
        inner_ids = train.index[kf_fold_num.to_numpy() == inner_fold]
        assert fit_ids.intersection(inner_ids).empty
        assert fit_ids.intersection(outer_ids).empty and inner_ids.intersection(outer_ids).empty
        assert fit_ids.intersection(val.index).empty and inner_ids.intersection(val.index).empty
        assert set(kf_groups.loc[fit_ids]).isdisjoint(set(kf_groups.loc[inner_ids]))
        assert set(kf_groups.loc[remaining_ids]).isdisjoint(set(kf_groups.loc[outer_ids]))
        print(f'B 제외 Fold={outer_fold} / 내부 검증 Fold={inner_fold} / 나머지 3개 학습', flush=True)
        # 기존 CatBoost 내부 회귀: 내부 검증 Fold에만 조기 종료 적용
        model_reg = CatBoostRegressor(**B_REG_PARAMS, cat_features=['station_id'])
        model_reg.fit(X_train_with_station.loc[fit_ids], y_train.loc[fit_ids],
                      eval_set=(X_train_with_station.loc[inner_ids], y_train.loc[inner_ids]),
                      early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)
        inner_pred.loc[inner_ids] = model_reg.predict(X_train_with_station.loc[inner_ids])
        tree_count = int(model_reg.tree_count_)
        inner_seen.loc[inner_ids] += 1
        b_reg_log.append(dict(stage='nested_residual', outer_fold=outer_fold,
                                  inner_fold=inner_fold, fit_rows=len(fit_ids), trees=tree_count))
    assert (inner_seen == 1).all() and np.isfinite(inner_pred.to_numpy()).all()
    assert inner_pred.index.equals(remaining_ids)
    b_nested_residuals[outer_fold] = (y_train.loc[remaining_ids] - inner_pred).rename('residual')
assert len(b_nested_residuals) == 5 and len(b_reg_log) == 20
print('B1 완료: 회귀 20회. 임계값만 변경할 때는 재실행하지 않습니다.')


### B2. Validation 분류기 정답용 회귀 OOF · 추가 5회
Train 전체의 고정 5-Fold 회귀 OOF 잔차를 별도로 준비합니다.
Validation 타깃은 사용하지 않으며, 이 잔차도 임계값 변경 시 재사용합니다.
기존 `train_kf_oof` 잔차는 새 CatBoost 분류기 정답으로 대입하지 않습니다.


In [ ]:
# B2. 독립 Validation 분류기 학습용 정답 잔차 생성; 회귀 5회 추가
assert b_cache_signature == data_signature
assert b_reg_snapshot == json.dumps(B_REG_PARAMS, sort_keys=True)
assert b_early_snapshot == EARLY_STOPPING_ROUNDS
b_full_oof = pd.Series(np.nan, index=train.index)
full_seen = pd.Series(0, index=train.index)
b_full_reg_log = []
for inner_fold in [1, 2, 3, 4, 5]:
    fit_ids = train.index[kf_fold_num.to_numpy() != inner_fold]
    inner_ids = train.index[kf_fold_num.to_numpy() == inner_fold]
    assert fit_ids.intersection(inner_ids).empty
    assert fit_ids.intersection(val.index).empty and inner_ids.intersection(val.index).empty
    assert set(kf_groups.loc[fit_ids]).isdisjoint(set(kf_groups.loc[inner_ids]))
    inner_pred = pd.Series(np.nan, index=inner_ids)
    # 기존 CatBoost 내부 회귀: 내부 검증 Fold에만 조기 종료 적용
    model_reg = CatBoostRegressor(**B_REG_PARAMS, cat_features=['station_id'])
    model_reg.fit(X_train_with_station.loc[fit_ids], y_train.loc[fit_ids],
                  eval_set=(X_train_with_station.loc[inner_ids], y_train.loc[inner_ids]),
                  early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)
    inner_pred.loc[inner_ids] = model_reg.predict(X_train_with_station.loc[inner_ids])
    tree_count = int(model_reg.tree_count_)
    b_full_oof.loc[inner_ids] = inner_pred
    full_seen.loc[inner_ids] += 1
    b_full_reg_log.append(dict(stage='validation_label_residual', inner_fold=inner_fold,
                                   fit_rows=len(fit_ids), trees=tree_count))
assert (full_seen == 1).all() and np.isfinite(b_full_oof.to_numpy()).all()
b_full_residual = (y_train - b_full_oof).rename('residual')
b_full_signature = (data_signature, json.dumps(B_REG_PARAMS, sort_keys=True), EARLY_STOPPING_ROUNDS)
print('B2 완료: 추가 회귀 5회. 회귀 성능 진단이 아닌 분류 정답 생성용입니다.')


### B3. 신규 변수 없는 기준 회귀 · 추가 1회
전체 Train으로 학습하고 Validation **예측값만** 준비합니다. eval_set/조기 종료는 없습니다.
동일 모델 설정으로 임계값별 실험 회귀와 비교합니다. 임계값이 바뀌어도 기준 예측은 재사용합니다.


In [ ]:
# B3. 기준 CatBoost 회귀 1회: Validation 정답 미사용
assert b_full_signature == (data_signature, json.dumps(B_REG_PARAMS, sort_keys=True), EARLY_STOPPING_ROUNDS)
b_baseline_model = CatBoostRegressor(**B_REG_PARAMS, cat_features=['station_id'], use_best_model=False)
b_baseline_model.fit(X_train_with_station, y_train, verbose=False)
b_baseline_pred = pd.Series(b_baseline_model.predict(X_val_with_station), index=val.index)
assert b_baseline_pred.index.equals(val.index) and np.isfinite(b_baseline_pred.to_numpy()).all()
b_baseline_signature = (data_signature, json.dumps(B_REG_PARAMS, sort_keys=True), EARLY_STOPPING_ROUNDS)
print('B3 기준 예측 준비 완료. 여기까지 공통 회귀 26회입니다.')


### B5. Fold별 분류 → 예측범주 → Validation 비교
각 Fold의 분류기는 B1의 **해당 Fold를 제외한 회귀 4회 결과**로 학습합니다.
5개 Fold의 예측범주는 `row_id` 순서로 합칩니다. 이후 B2의 전체 회귀 OOF 잔차로
별도 분류기를 학습하여 Validation 범주를 만듭니다.

표의 기본 범주 비율은 B2의 Train 전체 회귀 OOF 정답 분포이며, 외부 Fold별 학습 정답 분포와
Train/Validation 예측범주 분포도 각각 저장합니다. Macro F1·혼동행렬의 정답은
**Validation 관측값 − 해당 PART 기준 회귀 예측값**을 라벨링한 평가용 값입니다.
이 정답은 모델 입력으로 사용하지 않습니다. 없는 범주도 포함한 고정 3범주 Macro F1입니다.


In [ ]:
# B5. 회귀 잔차는 재사용; 분류/실험 회귀만 임계값마다 학습
# 데이터 또는 설정을 바꿨으면 공통 4와 B1~B3을 먼저 다시 실행
current_signature = hashlib.sha256(
    pd.util.hash_pandas_object(X_train_with_station, index=True).to_numpy().tobytes()
    + pd.util.hash_pandas_object(y_train, index=True).to_numpy().tobytes()
    + pd.util.hash_pandas_object(kf_fold_num, index=True).to_numpy().tobytes()
    + pd.util.hash_pandas_object(X_val_with_station, index=True).to_numpy().tobytes()
).hexdigest()
assert current_signature == data_signature == b_cache_signature, '데이터 변경: 준비 셀 재실행 필요'
assert b_reg_snapshot == json.dumps(B_REG_PARAMS, sort_keys=True)
assert b_early_snapshot == EARLY_STOPPING_ROUNDS
assert b_full_signature == b_baseline_signature == (data_signature, json.dumps(B_REG_PARAMS, sort_keys=True), EARLY_STOPPING_ROUNDS)
assert train.index.equals(original_row_index)
assert np.array_equal(kf_fold_num.to_numpy(), original_fold_num.to_numpy())
assert np.array_equal(kf_fold_num.to_numpy(), train['event_id'].map(kf_fold_map.set_index('event_id')['fold_num']))

# 매 실행마다 새 폴더: 임계값을 바꿔도 과거 결과를 덮어쓰지 않음
b_dir = OUTPUT_ROOT / ('PART_B_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8])
b_dir.mkdir(parents=True, exist_ok=False)
(b_dir / 'settings.json').write_text(json.dumps(dict(
    thresholds=[float(final_threshold)], regression_params=B_REG_PARAMS, classifier_params=CAT_CLS_PARAMS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS, data_signature=data_signature,
    versions={p: importlib.metadata.version(p) for p in ['pandas','scikit-learn','lightgbm','catboost']}),
    ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame(b_reg_log + b_full_reg_log).to_csv(b_dir / 'regression_training_log.csv', index=False)
b_results = []
b_classification_log = []
for threshold in [final_threshold]:
    # B는 단일 확정값 하나만 수행; A만 후보를 반복 비교
    print(f'PART B: threshold={threshold}', flush=True)
    try:
        threshold_dir = b_dir / f'threshold_{threshold}'
        threshold_dir.mkdir(exist_ok=False)
        train_category = pd.Series(index=train.index, dtype=object, name=NEW_FEATURE)
        category_seen = pd.Series(0, index=train.index)
        fold_distributions = []
        for outer_fold in [1, 2, 3, 4, 5]:
            # 이 Fold의 분류 정답은 B1에서 이 Fold를 제외한 내부 회귀 4회로 생성됨
            predict_ids = train.index[kf_fold_num.to_numpy() == outer_fold]
            fit_ids = train.index[kf_fold_num.to_numpy() != outer_fold]
            residual = b_nested_residuals[outer_fold]
            assert residual.index.equals(fit_ids) and np.isfinite(residual.to_numpy()).all()
            assert fit_ids.intersection(predict_ids).empty and fit_ids.intersection(val.index).empty
            assert set(kf_groups.loc[fit_ids]).isdisjoint(set(kf_groups.loc[predict_ids]))
            residual_label = pd.Series(np.where(residual > threshold, 'under',
                np.where(residual < -threshold, 'over', 'normal')), index=fit_ids, name='residual_label')
            counts = residual_label.value_counts().reindex(CLASSES, fill_value=0)
            for label in CLASSES:
                fold_distributions.append(dict(fold_num=outer_fold, label=label,
                    count=int(counts[label]), ratio=float(counts[label] / len(fit_ids))))
            # 학습 정답이 한 범주라면 분류 fit을 건너뛰고 상수 예측, 실제 횟수 기록
            class_count = residual_label.nunique()
            if class_count == 1:
                warnings.warn(f'PART B Fold {outer_fold}: 한 범주만 존재, 상수 예측')
                predicted = np.repeat(residual_label.iloc[0], len(predict_ids))
            else:
                if class_count < 3:
                    warnings.warn(f'PART B Fold {outer_fold}: 학습 정답에 일부 범주가 없습니다.')
                model_cls = CatBoostClassifier(**CAT_CLS_PARAMS, cat_features=['station_id'])
                model_cls.fit(X_train_with_station.loc[fit_ids], residual_label)
                predicted = np.asarray(model_cls.predict(X_train_with_station.loc[predict_ids])).reshape(-1).astype(str)
            predicted_category = pd.Series(predicted, index=predict_ids, name=NEW_FEATURE)
            assert set(predicted_category.unique()).issubset(CLASSES)
            train_category.loc[predict_ids] = predicted_category
            category_seen.loc[predict_ids] += 1
            b_classification_log.append(dict(threshold=float(threshold), fold=outer_fold,
                                                   fit_performed=class_count > 1))
        assert (category_seen == 1).all() and train_category.notna().all()
        assert train_category.index.equals(train.index) and train_category.name == NEW_FEATURE

        # Validation용 별도 분류기: B2의 Train 전체 회귀 OOF 잔차를 정답으로 사용
        residual = b_full_residual
        assert residual.index.equals(train.index) and np.isfinite(residual.to_numpy()).all()
        residual_label = pd.Series(np.where(residual > threshold, 'under',
            np.where(residual < -threshold, 'over', 'normal')), index=train.index, name='residual_label')
        counts = residual_label.value_counts().reindex(CLASSES, fill_value=0)
        distribution = pd.DataFrame({'count': counts, 'ratio': counts / len(train)})
        display(distribution.rename(index={'under':'과소예측','normal':'일반오차','over':'과대예측'}))
        class_count = residual_label.nunique()
        if class_count == 1:
            warnings.warn('Validation용 분류기 정답이 한 범주뿐입니다. 상수 예측 사용.')
            val_predicted = np.repeat(residual_label.iloc[0], len(val))
        else:
            if class_count < 3:
                warnings.warn('Validation용 분류기 학습 정답에 일부 범주가 없습니다.')
            model_val_cls = CatBoostClassifier(**CAT_CLS_PARAMS, cat_features=['station_id'])
            model_val_cls.fit(X_train_with_station, residual_label)
            val_predicted = np.asarray(model_val_cls.predict(X_val_with_station)).reshape(-1).astype(str)
        b_classification_log.append(dict(threshold=float(threshold), fold='Validation',
                                              fit_performed=class_count > 1))
        val_category = pd.Series(val_predicted, index=val.index, name=NEW_FEATURE)
        assert val_category.notna().all() and set(val_category.unique()).issubset(CLASSES)
        assert val_category.index.equals(val.index) and val_category.name == NEW_FEATURE

        # 기존 32개 변수 + 분류 predict 결과 1개만 사용. residual_label은 입력하지 않음
        assert train_category.name == val_category.name == NEW_FEATURE
        X_exp_train = X_train_with_station.copy()
        X_exp_val = X_val_with_station.copy()
        X_exp_train[NEW_FEATURE] = train_category.astype(str)
        X_exp_val[NEW_FEATURE] = val_category.astype(str)
        model_exp = CatBoostRegressor(**B_REG_PARAMS, cat_features=['station_id', NEW_FEATURE], use_best_model=False)
        model_exp.fit(X_exp_train, y_train, verbose=False)
        experiment_pred = pd.Series(model_exp.predict(X_exp_val), index=val.index)
        assert X_exp_train.columns.tolist() == num_cols_fe + ['station_id', NEW_FEATURE]
        assert X_exp_val.columns.equals(X_exp_train.columns)
        assert X_exp_train.index.equals(y_train.index)
        assert experiment_pred.index.equals(val.index) and np.isfinite(experiment_pred.to_numpy()).all()

        # 모든 fit/predict 후에만 Validation 정답을 평가에 사용
        y_val = val[target_col]
        assert np.isfinite(y_val.to_numpy()).all()
        baseline_rmse = float(np.sqrt(mean_squared_error(y_val, b_baseline_pred)))
        experiment_rmse = float(np.sqrt(mean_squared_error(y_val, experiment_pred)))
        eval_residual = y_val - b_baseline_pred
        eval_label = pd.Series(np.where(eval_residual > threshold, 'under',
            np.where(eval_residual < -threshold, 'over', 'normal')), index=val.index)
        score_f1 = float(f1_score(eval_label, val_category, labels=CLASSES, average='macro', zero_division=0))
        matrix = pd.DataFrame(confusion_matrix(eval_label, val_category, labels=CLASSES), index=CLASSES, columns=CLASSES)
        print('Validation 혼동행렬: 행=정답, 열=예측')
        display(matrix)
        b_results.append(dict(threshold=float(threshold), under_count=int(counts['under']),
            normal_count=int(counts['normal']), over_count=int(counts['over']),
            under_ratio=float(distribution.loc['under','ratio']), normal_ratio=float(distribution.loc['normal','ratio']),
            over_ratio=float(distribution.loc['over','ratio']), macro_f1=score_f1,
            baseline_rmse=baseline_rmse, experiment_rmse=experiment_rmse,
            delta_rmse=experiment_rmse-baseline_rmse, improvement_rmse=baseline_rmse-experiment_rmse,
            improvement_pct=100*(baseline_rmse-experiment_rmse)/baseline_rmse if baseline_rmse else np.nan))
        # row_id 유지; 정답 분포와 예측 분포를 구분해 저장
        distribution.to_csv(threshold_dir / 'train_full_oof_label_distribution.csv')
        pd.DataFrame(fold_distributions).to_csv(threshold_dir / 'fold_training_label_distribution.csv', index=False)
        pd.DataFrame({'fold_num':kf_fold_num, NEW_FEATURE:train_category}).to_parquet(threshold_dir / 'train_classification_oof.parquet')
        pd.DataFrame({'y_true':y_val, 'baseline_pred':b_baseline_pred, 'experiment_pred':experiment_pred,
                       NEW_FEATURE:val_category, 'evaluation_label':eval_label}).to_parquet(threshold_dir / 'validation_predictions.parquet')
        # 범주별 데이터 수/비율: Train 예측, Validation 예측, Validation 평가 정답 각각 기록
        for name, labels in [('train_predicted',train_category), ('validation_predicted',val_category), ('validation_truth',eval_label)]:
            class_counts = labels.value_counts().reindex(CLASSES, fill_value=0)
            pd.DataFrame({'count':class_counts, 'ratio':class_counts/len(labels)}).to_csv(threshold_dir / f'{name}_distribution.csv')
        matrix.to_csv(threshold_dir / 'validation_confusion_matrix.csv')
        pd.DataFrame(b_results).to_csv(b_dir / 'validation_results.csv', index=False)
        pd.DataFrame(b_classification_log).to_csv(b_dir / 'classification_training_log.csv', index=False)
    except Exception as exc:
        (b_dir / 'error.txt').write_text(f'threshold={threshold}\n{type(exc).__name__}: {exc}', encoding='utf-8')
        raise
b_results_df = pd.DataFrame(b_results)
display(b_results_df)
assert train.index.equals(original_row_index)
assert np.array_equal(kf_fold_num.to_numpy(), original_fold_num.to_numpy())
print('ΔRMSE=실험−기준: 음수이면 개선. improvement_rmse=기준−실험: 양수이면 개선.')
print('결과 폴더:', b_dir)


## 학습 횟수와 재실행
| 구분 | 핵심 | 별도 추가 | 총합 |
|---|---:|---:|---:|
| A 단일 임계값 | 회귀20 + 분류5 + 실험회귀1 = 26 | Validation 정답회귀5 + 분류1 + 기준회귀1 = 7 | 33 |
| A m개 (잔차/기준 재사용) | 20+6m | 6+m | **26+7m** |
| A 초기 2개 | 32 | 8 | **40** |
| B 단일 확정 임계값 | 26 | 7 | **33** |

A는 회귀 **26+m**, 분류 **6m**. B는 회귀 **27**, 분류 **6**입니다.
A 초기 후보 2개 + B 1개를 모두 실행하면 총 73회입니다.
한 범주만 남으면 해당 분류 fit 대신 상수 예측하므로 실제 fit 횟수는 줄어들며 로그로 구별합니다.
A에서 준비 완료 후 새 임계값 m개를 실행하면 **추가 7m회**만 학습합니다.

- 임계값만 변경: **A4 → A5 → A6**. B의 값만 변경한다면 **B0 → B5**.
- 데이터·Fold·독립변수·회귀 설정·조기 종료 설정 변경: 공통 준비 확인 후 해당 PART 1~3부터 재실행.
- A와 B는 각자의 잔차와 기준 예측을 메모리에 보관하며 서로 공유하지 않음.
- `normal`은 일반오차(−t ≤ 잔차 ≤ t). 분류 예측범주만 회귀 입력으로 사용.
- 내부 회귀의 내부 검증 정답은 조기 종료에 사용됨. 외부 예측 Fold와 독립 Validation 정답은 fit/조기 종료에 사용하지 않음.
- 외부 Fold별 본 회귀 성능 진단, Test 예측, 제출파일 생성, 임계값 자동 선정은 없음.
- 이번 수정에서는 실제 모델을 학습하지 않음. 정적 검증만 완료한 코드이며 실행 결과는 사용자가 확인.
